In [1]:
from dotenv import load_dotenv
import os
load_dotenv()  # .env 파일 로드
api_key = os.getenv("IBMQ_API_KEY")  # from .env file

from qiskit_ibm_runtime import QiskitRuntimeService
 
service = QiskitRuntimeService(channel="ibm_quantum", token=api_key)

In [5]:
from qiskit_ibm_runtime import QiskitRuntimeService
 
# Save an IBM Quantum account and set it as your default account.
QiskitRuntimeService.save_account(channel="ibm_quantum", token=api_key, set_as_default=True, overwrite=True)
 
# Load saved credentials
service = QiskitRuntimeService()

In [6]:
service.backends()

[<IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_kyiv')>,
 <IBMBackend('ibm_sherbrooke')>]

In [8]:
 from qiskit import QuantumCircuit
 from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
 
 # Create empty circuit
 example_circuit = QuantumCircuit(2)
 example_circuit.measure_all()
 
 # You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
 service = QiskitRuntimeService()
 backend = service.backend("ibm_sherbrooke")
 job = Sampler(backend).run([example_circuit])
 print(f"job id: {job.job_id()}")
 result = job.result()
 print(result)

job id: cz9zxs7tp60g008h10wg
PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([{'__type__': 'DoubleSliceSpan', '__value__': {'start': datetime.datetime(2025, 3, 14, 10, 3, 20, 126977), 'stop': datetime.datetime(2025, 3, 14, 10, 3, 23, 213799), 'data_slices': {'0': [[4096], 0, 1, 0, 4096]}}}])}, 'version': 2})


In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.visualization import plot_histogram
import torch
import numpy as np
from modules.utils import read_args
import os 

def QGAN2_ibmq(circuit, inputs, params):
    assert(len(inputs) == params.shape[1])
    n_qubits = len(inputs)
    n_layers = params.shape[0]

    for i in range(n_qubits):
        circuit.ry(inputs[i] * np.pi/2, i)
    for l in range(n_layers):
        for i in range(n_qubits):
            circuit.ry(params[l, i, 0], i)
        if l < n_layers-1:
            for i in range(n_qubits-1):
                circuit.cx(i, i+1)
            circuit.cx(n_qubits-1, 0)

In [14]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches


epoch = 288
trial_name = "False_np5_nl10_biased_diamond_False_Mar08_18_52_18"
base_dir = os.path.join('./정리/Scientific Reports/2D', trial_name)
args_file_path = os.path.join(base_dir, 'args.txt')
param_file_path = os.path.join(base_dir, 'params', f'generator_params_epoch{epoch}.pth')
generator_params = torch.load(param_file_path, weights_only=True).detach().numpy()
n_qubits, code_qubits, n_layers, SEED = read_args(args_file_path, "n_qubits", "code_qubits", "n_layers", "seed")
print(f"n_qubits: {n_qubits}, n_layers: {n_layers}, SEED: {SEED}")

output_records = []
input_records = []

rep = 1000

backend = GenericBackendV2(num_qubits=n_qubits)

postprocessing_dir = os.path.join(base_dir, 'postprocessing')
os.makedirs(postprocessing_dir, exist_ok=True)
output_img_dir = os.path.join(base_dir, 'postprocessing', 'ibmq_simulator')
os.makedirs(output_img_dir, exist_ok=True)
output_file_path = os.path.join(postprocessing_dir, 'ibmq_simulator', 'ibmq_simulator_outputs.txt')
codes_file_path = os.path.join(postprocessing_dir, 'ibmq_simulator', 'ibmq_simulator_codes.txt')

def bitwise_sums(arr):
    n = len(arr).bit_length() - 1  # 비트 길이를 계산하여 반복 횟수를 정함
    sums = torch.zeros(n)  # 결과를 저장할 텐서
    for bit in range(n):
        # 조건에 맞는 인덱스 선택을 위해 i-th 비트를 검사
        mask = (torch.arange(len(arr)) >> bit) & 1
        sums[bit] = arr[mask.bool()].sum()  # 조건에 맞는 원소들의 합산
    return sums

for i in tqdm(range(rep)):
    circuit = QuantumCircuit(n_qubits)
    z = np.random.uniform(-SEED, SEED, (n_qubits)) # input
    QGAN2_ibmq(circuit, z, generator_params)
    circuit.measure_all()
    
    transpiled_circuit = transpile(circuit, backend)
    # Run the transpiled circuit using the simulated backend
    job = backend.run(transpiled_circuit)
    counts = job.result().get_counts()
    arr = np.zeros(2**n_qubits, dtype=int)
    for bitstr, cnt in counts.items():
        arr[int(bitstr, 2)] = cnt
    probabilities = arr / arr.sum()
    probabilities = bitwise_sums(probabilities)
    #print("i = ", i, probabilities )
    
    output_records.append(probabilities)
    input_records.append(z[-code_qubits:])
    if i % 200 != 199:
        continue

    outputs = np.array(output_records)
    inputs = np.array(input_records)

    # 시각화
    for code_ind in range(code_qubits):
        plt.figure(figsize=(12,10))
        plt.scatter(outputs[:, 0], outputs[:, 1], c=inputs[:, code_ind], cmap='RdYlBu', alpha=0.4, s=10)
        plt.colorbar()  # 색상 막대 추가
        plt.title(f'code{code_ind} (size={i+1}, ibmq simulator)')
        plt.xlim((0, 1))
        plt.ylim((0, 1))
        ax = plt.gca()
        
        # 중심 (0.6, 0.6), 팔 길이 0.2sqrt(2)인 다이아몬드 추가
        arm = 0.2 * np.sqrt(2)
        circle = patches.Polygon([[0.6+arm, 0.6], [0.6, 0.6-arm], [0.6-arm, 0.6], [0.6, 0.6+arm]], closed=True, fill=False, edgecolor='red')
        ax.add_patch(circle)

        save_dir = os.path.join(output_img_dir, f'ibmq_simulator_code{code_ind}_{i+1}.png')
        plt.savefig(save_dir)
        plt.close()

np.savetxt(output_file_path, outputs)
np.savetxt(codes_file_path, inputs)

n_qubits: 5, n_layers: 10, SEED: 0.5


  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [01:08<00:00, 14.54it/s]


In [15]:
import pandas as pd
import ndtest # 2D 분포 검정에 사용
gen_outputs = outputs
gen_codes = inputs[:, -code_qubits:]

# code와 x, y의 상관관계를 측정 후 기록
df = pd.DataFrame({'x': gen_outputs[:, 0], 'y': gen_outputs[:, 1]})
for i in range(code_qubits):
    df[f'code{i}'] = gen_codes[:, i]
corr_mat = df.corr().to_numpy()
writer = {}
corr_mat = df.corr().to_numpy()
for i in range(code_qubits):
    writer[f'Corr/code{i}-x'] = corr_mat[0, i+2]
    writer[f'Corr/code{i}-y'] = corr_mat[1, i+2]

cos_theta = (corr_mat[0, 2] * corr_mat[0, 3] + corr_mat[1, 2] * corr_mat[1, 3]) / (
    np.sqrt(corr_mat[0, 2]**2 + corr_mat[1, 2]**2) * np.sqrt(corr_mat[0, 3]**2 + corr_mat[1, 3]**2)
)
theta_degrees = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))

# 예각으로 변환
theta_degrees = min(theta_degrees, 180 - theta_degrees)
writer['angle']=theta_degrees

train_in = np.loadtxt(f'data/2D/biased_diamond_1000_1.txt')
p_value, D_ks = ndtest.ks2d2s(gen_outputs[:, 0], gen_outputs[:, 1], train_in[:, 0], train_in[:, 1], extra=True)

writer['p_value'] = p_value
writer['D_ks'] = D_ks

writer['code0-norm'] = np.sqrt(writer['Corr/code0-x']**2 + writer['Corr/code0-y']**2)
writer['code1-norm'] = np.sqrt(writer['Corr/code1-x']**2 + writer['Corr/code1-y']**2)


In [18]:
import json
with open(os.path.join(postprocessing_dir, 'ibmq_simulator', 'ibmq_simulator_writer.json'), 'w') as f:
    json.dump(writer, f, indent=4)
writer

{'Corr/code0-x': 0.24708296701623908,
 'Corr/code0-y': 0.016533609929238954,
 'Corr/code1-x': 0.5127212273585161,
 'Corr/code1-y': -0.06751442156676762,
 'angle': 11.329723343437193,
 'p_value': 1.6559133757733244e-18,
 'D_ks': 0.24800000000000005,
 'code0-norm': 0.24763552420208235,
 'code1-norm': 0.5171472267193534}